# FinTrust Data Science Track — Week 2

This notebook prepares the modelling dataset, runs EDA, creates engineered features, and trains a baseline logistic-regression model.


In [ ]:
import json
from pathlib import Path

import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import joblib


In [ ]:
base = Path(".")
customer = pd.read_csv(base / "customer.csv")
transactions = pd.read_csv(base / "transactions.csv")
transactions["Transaction_DateTime"] = pd.to_datetime(transactions["Transaction_DateTime"])
transactions["Risk_Review_Flag"] = transactions["Risk_Review_Flag"].map({"Yes": 1, "No": 0})
merged = transactions.merge(customer, on="Customer_ID", how="left", validate="many_to_one")
merged["Transaction_Hour"] = merged["Transaction_DateTime"].dt.hour
merged["Transaction_Day_of_Week"] = merged["Transaction_DateTime"].dt.dayofweek
merged["Amount_Log"] = merged["Amount_NGN"].apply(lambda x: math.log(x) if pd.notna(x) and x > 0 else 0)
merged["Is_International"] = (merged["International_Transaction"] == "Yes").astype(int)
merged["Is_Failed"] = (merged["Transaction_Status"] == "Failed").astype(int)
merged["Is_Reversed"] = (merged["Transaction_Status"] == "Reversed").astype(int)
merged["Is_Pending"] = (merged["Transaction_Status"] == "Pending").astype(int)
merged["Customer_Transaction_Count"] = merged.groupby("Customer_ID")["Transaction_ID"].transform("count")
merged["Customer_Avg_Amount"] = merged.groupby("Customer_ID")["Amount_NGN"].transform("mean")
merged["Amount_to_Customer_Avg"] = merged["Amount_NGN"] / merged["Customer_Avg_Amount"]
merged["Digital_Engagement_Band"] = pd.cut(merged["Digital_Engagement_Score"], bins=[0, 40, 60, 80, 100], labels=["Low", "Moderate", "High", "Very High"])
merged["Tenure_Bucket"] = pd.cut(merged["Tenure_Months"], bins=[0, 12, 24, 48, 84, 200], labels=["0-12", "13-24", "25-48", "49-84", "85+"])
merged["Amount_Band"] = pd.cut(merged["Amount_NGN"], bins=[0, 1000, 10000, 50000, 100000, 1_000_000_000], labels=["<1k", "1k-10k", "10k-50k", "50k-100k", "100k+"])
prepared = merged.copy()
prepared.to_csv("prepared_modelling_dataset.csv", index=False)


## EDA: Risk patterns by key dimensions


In [ ]:
sns.set_theme(style="whitegrid", context="talk")
fig, ax = plt.subplots(figsize=(7, 5))
value_counts = prepared["Risk_Review_Flag"].value_counts().sort_index()
ax.bar(["No Risk Review", "Risk Review"], value_counts.values, color=["#4C78A8", "#F58518"])
ax.set_title("Target Distribution")
plt.tight_layout(); fig.savefig("outputs/01_target_distribution.png", dpi=200); plt.close(fig)
fig, ax = plt.subplots(figsize=(10, 6))
per_type = prepared.groupby("Transaction_Type")["Risk_Review_Flag"].mean().sort_values(ascending=False)
ax.bar(per_type.index, per_type.values * 100, color=sns.color_palette("viridis", len(per_type)))
ax.set_title("Risk Review Rate by Transaction Type")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); fig.savefig("outputs/02_transaction_type_risk_rate.png", dpi=200); plt.close(fig)
fig, ax = plt.subplots(figsize=(8, 5))
status_rate = prepared.groupby("Transaction_Status")["Risk_Review_Flag"].mean().sort_values(ascending=False)
ax.bar(status_rate.index, status_rate.values * 100)
ax.set_title("Risk Review Rate by Status")
plt.tight_layout(); fig.savefig("outputs/03_status_risk_rate.png", dpi=200); plt.close(fig)


## Model training


In [ ]:
feature_cols = [
    "Transaction_Type", "Channel", "Device_Type", "Location", "Transaction_Status",
    "Customer_Segment", "Account_Type", "Gender", "Preferred_Channel", "Account_Status",
    "Amount_NGN", "Amount_Log", "Transaction_Hour", "Transaction_Day_of_Week",
    "Is_International", "Is_Failed", "Is_Reversed", "Is_Pending",
    "Customer_Transaction_Count", "Customer_Avg_Amount", "Amount_to_Customer_Avg",
    "Digital_Engagement_Score", "Tenure_Months", "Digital_Engagement_Band", "Tenure_Bucket", "Amount_Band"
]
X = prepared[feature_cols].copy()
y = prepared["Risk_Review_Flag"]
for c in X.columns:
    if pd.api.types.is_numeric_dtype(X[c]):
        X[c] = X[c].fillna(X[c].median())
    else:
        X[c] = X[c].astype(str).fillna("Missing").replace("nan", "Missing")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced"))
])
model.fit(X_train, y_train)
preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]
metrics = {
    "accuracy": round(float(accuracy_score(y_test, preds)), 4),
    "precision": round(float(precision_score(y_test, preds, zero_division=0)), 4),
    "recall": round(float(recall_score(y_test, preds, zero_division=0)), 4),
    "f1_score": round(float(f1_score(y_test, preds, zero_division=0)), 4),
    "roc_auc": round(float(roc_auc_score(y_test, probs)), 4),
}
metrics
